In [0]:
kafka_bootstrap = "project-kafka-26-project-193d.c.aivencloud.com:17885"
kafka_topic = "reddit_posts"
ca_cert = "/Volumes/workspace/default/reddit_volume/ca.pem"

kafka_username = "avnadmin"
kafka_password = "<kafka_password>"

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option(
        "kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username="{kafka_username}" password="{kafka_password}";'
    )
    .option("kafka.ssl.truststore.type", "PEM")
    .option("kafka.ssl.truststore.location", ca_cert)
    .load()
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

bronze_df = (
    kafka_df
    .select(
        col("value").cast("string").alias("raw_json"),
        col("topic").alias("kafka_topic"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        current_timestamp().alias("ingestion_timestamp")
    )
)

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/reddit_volume/checkpoints/reddit_bronze"
    )
    .toTable("workspace.reddit.reddit_bronze")
)

query.awaitTermination()

26/08/15 04:24:15 Spark Server has not sent updates for Streaming Query 0bd315d6-75ab-4c55-b501-95010780c895 in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 0b251b24-26ee-4f05-b0d1-22110e53a1ab. This is typically not a problem.
26/08/15 04:24:40 Spark Server has not sent updates for Streaming Query 0bd315d6-75ab-4c55-b501-95010780c895 in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 0b251b24-26ee-4f05-b0d1-22110e53a1ab. This is typically not a problem.
26/08/15 04:25:05 Spark Server has not sent updates for Streaming Query 0bd315d6-75ab-4c55-b501-95010780c895 in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 0b251b24-26ee-4f05-b0d1-22110e53a1ab. This is typically not a problem.


In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.reddit.reddit_bronze;

total_records
3121


In [0]:
%sql
SELECT raw_json
FROM workspace.reddit.reddit_bronze
LIMIT 1;

raw_json
"{""approved_at_utc"": null, ""subreddit"": ""SwordAndSupperGame"", ""selftext"": ""This post contains content not supported on old Reddit. [Click here to view the full post](https://sh.reddit.com/r/SwordAndSupperGame/comments/1vny7mf)"", ""author_fullname"": ""t2_4adj04u"", ""saved"": false, ""mod_reason_title"": null, ""gilded"": 0, ""clicked"": false, ""title"": ""1 or 2 \u2b50\ufe0f \ud83c\udf89 Lvl 401!!! Max Gold Drop \ud83c\udf89"", ""link_flair_richtext"": [], ""subreddit_name_prefixed"": ""r/SwordAndSupperGame"", ""hidden"": false, ""pwls"": 6, ""link_flair_css_class"": null, ""downs"": 0, ""thumbnail_height"": null, ""top_awarded_type"": null, ""hide_score"": true, ""name"": ""t3_1vny7mf"", ""quarantine"": false, ""link_flair_text_color"": ""dark"", ""upvote_ratio"": 1.0, ""author_flair_background_color"": null, ""subreddit_type"": ""public"", ""ups"": 1, ""total_awards_received"": 0, ""media_embed"": {}, ""thumbnail_width"": null, ""author_flair_template_id"": null, ""is_original_content"": false, ""user_reports"": [], ""secure_media"": null, ""is_reddit_media_domain"": false, ""is_meta"": false, ""category"": null, ""secure_media_embed"": {}, ""link_flair_text"": null, ""can_mod_post"": false, ""score"": 1, ""approved_by"": null, ""is_created_from_ads_ui"": false, ""author_premium"": false, ""thumbnail"": ""self"", ""edited"": false, ""author_flair_css_class"": null, ""author_flair_richtext"": [], ""gildings"": {}, ""content_categories"": null, ""is_self"": true, ""mod_note"": null, ""created"": 1786684810.0, ""link_flair_type"": ""text"", ""wls"": 6, ""removed_by_category"": null, ""banned_by"": null, ""author_flair_type"": ""text"", ""domain"": ""self.SwordAndSupperGame"", ""allow_live_comments"": false, ""selftext_html"": ""<!-- SC_OFF --><div class=\""md\""><p>This post contains content not supported on old Reddit. <a href=\""https://sh.reddit.com/r/SwordAndSupperGame/comments/1vny7mf\"">Click here to view the full post</a></p>\n</div><!-- SC_ON -->"", ""likes"": null, ""suggested_sort"": null, ""banned_at_utc"": null, ""view_count"": null, ""archived"": false, ""no_follow"": true, ""is_crosspostable"": true, ""pinned"": false, ""over_18"": false, ""all_awardings"": [], ""awarders"": [], ""media_only"": false, ""can_gild"": false, ""spoiler"": false, ""locked"": false, ""author_flair_text"": null, ""treatment_tags"": [], ""visited"": false, ""removed_by"": null, ""num_reports"": null, ""distinguished"": null, ""subreddit_id"": ""t5_eimoap"", ""author_is_blocked"": false, ""mod_reason_by"": null, ""removal_reason"": null, ""link_flair_background_color"": """", ""id"": ""1vny7mf"", ""is_robot_indexable"": true, ""report_reasons"": null, ""author"": ""Sashimi_Rollin_"", ""discussion_type"": null, ""num_comments"": 0, ""send_replies"": true, ""contest_mode"": false, ""mod_reports"": [], ""author_patreon_flair"": false, ""author_flair_text_color"": null, ""permalink"": ""/r/SwordAndSupperGame/comments/1vny7mf/1_or_2_lvl_401_max_gold_drop/"", ""stickied"": false, ""url"": ""https://www.reddit.com/r/SwordAndSupperGame/comments/1vny7mf/1_or_2_lvl_401_max_gold_drop/"", ""subreddit_subscribers"": 231064, ""created_utc"": 1786684810.0, ""num_crossposts"": 0, ""media"": null, ""is_video"": false}"
